In [0]:
import os
from pyspark.sql.functions import current_timestamp, lit, input_file_name

BASE_PATH   = "/Workspace/Users/darwinramesh2110@gmail.com/Databricks_dbt_project/Datasets/dataset_1"
CATALOG     = "workspace"   
SCHEMA      = "bronze"

tables = {
    "olist_orders_dataset.csv":                  "orders",
    "olist_order_items_dataset.csv":             "order_items",
    "olist_order_payments_dataset.csv":          "order_payments",
    "olist_order_reviews_dataset.csv":           "order_reviews",
    "olist_customers_dataset.csv":               "customers",
    "olist_products_dataset.csv":                "products",
    "olist_sellers_dataset.csv":                 "sellers",
    "olist_geolocation_dataset.csv":             "geolocation",
    "product_category_name_translation.csv":     "product_category_translation",
}
print("Prepared tables for ingesting")

In [0]:
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Schema is ready : {CATALOG}.{SCHEMA}")

In [0]:
def ingest_csv_to_bronze(filename, table_name):
    source_path  = f"{BASE_PATH}/{filename}"
    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"

    print(f"\n→ Ingesting: {filename}")
    print(f"  Target   : {target_table}")

    df = (
        spark.read
            .format("csv")
            .option("header", "true")
            .option("inferSchema", "true")
            .load(source_path)
            .withColumn("_ingested_at", current_timestamp())
            .withColumn("_source_file", lit(filename))
            .withColumn("_layer",       lit("bronze"))
    )

    df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)

    count = spark.table(target_table).count()
    print(f"Done — {count:,} rows written to {target_table}")


In [0]:
for filename, table_name in tables.items():
    ingest_csv_to_bronze(filename, table_name)

print("\nBronze ingestion complete.")

In [0]:
spark.sql("SELECT current_catalog()").show()

In [0]:
print(f"BRONZE LAYER — ROW COUNT SUMMARY")

for filename, table_name in tables.items():
    target_table = f"{CATALOG}.{SCHEMA}.{table_name}"
    try:
        count = spark.table(target_table).count()
        print(f"  {table_name:<40} {count:>10,} rows")
    except Exception as e:
        print(f"  {table_name:<40} ERROR: {e}")

print("Autoloader script completed.")